In [1]:
import os
os.environ["GOOGLE_CLOUD_PROJECT"] = "yellow-taxi-trips-2026"

In [2]:
#On charge l'extension BigQuery pour le notebook
%load_ext google.cloud.bigquery 

c:\Users\EDITH ADJE\Documents\GCP Project\.venv\Lib\site-packages\google\cloud\bigquery\__init__.py:239: FutureWarning: %load_ext google.cloud.bigquery is deprecated. Install bigquery-magics package and use `%load_ext bigquery_magics`, instead.
  warnings.warn(


-- III/ Financial & Pricing Analysis

-- Question 7: How does the total fare revenue of yellow taxis change over time?

In [3]:
%%bigquery
CREATE OR REPLACE VIEW `views_fordashboard.total_fare_revenue_over_time` AS
SELECT 
    DATE(t.tpep_pickup_datetime) AS trip_date,
    EXTRACT(YEAR FROM t.tpep_pickup_datetime) AS year,
    EXTRACT(MONTH FROM t.tpep_pickup_datetime) AS month,
    EXTRACT(WEEK FROM t.tpep_pickup_datetime) AS week,
    EXTRACT(DAYOFWEEK FROM t.tpep_pickup_datetime) AS weekday,
    SUM(t.total_amount) AS total_revenue,
    SUM(t.fare_amount) AS fare_revenue,
    SUM(t.tip_amount) AS tip_revenue,
    SUM(t.tolls_amount) AS tolls_revenue,
    SUM(t.congestion_surcharge) AS congestion_revenue
FROM `yellow-taxi-trips-2026.transformed_data.cleaned_and_filtered` t
GROUP BY trip_date, year, month, week, weekday;

c:\Users\EDITH ADJE\Documents\GCP Project\.venv\Lib\site-packages\google\cloud\bigquery\job\query.py:2170: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  query_result = wait_for_query(self, progress_bar_type, max_results=max_results)


""


In [ ]:
%%bigquery
SELECT * FROM `views_fordashboard.total_fare_revenue_over_time` LIMIT 100;

-- Question 8: What is the average fare per trip, and how does it vary by borough, time of day, and trip distance?

In [4]:
%%bigquery
CREATE OR REPLACE VIEW `views_fordashboard.avg_fare_analysis` AS
SELECT 
    DATE(t.tpep_pickup_datetime) AS trip_date,
    EXTRACT(YEAR FROM t.tpep_pickup_datetime) AS year,
    EXTRACT(MONTH FROM t.tpep_pickup_datetime) AS month,
    EXTRACT(HOUR FROM t.tpep_pickup_datetime) AS pickup_hour,
    pz.Borough AS pickup_borough,
    dz.Borough AS dropoff_borough,
    ROUND(AVG(t.fare_amount), 2) AS avg_fare_per_trip,
    ROUND(AVG(t.total_amount), 2) AS avg_total_amount_per_trip,
    ROUND(AVG(t.trip_distance), 2) AS avg_trip_distance,
    COUNT(*) AS total_trips
FROM `yellow-taxi-trips-2026.transformed_data.cleaned_and_filtered` t
JOIN `yellow-taxi-trips-2026.raw_yellowtrips.taxi_zone` pz 
    ON t.PULocationID = pz.LocationID
JOIN `yellow-taxi-trips-2026.raw_yellowtrips.taxi_zone` dz 
    ON t.DOLocationID = dz.LocationID
GROUP BY trip_date, year, month, pickup_hour, pickup_borough, dropoff_borough;

c:\Users\EDITH ADJE\Documents\GCP Project\.venv\Lib\site-packages\google\cloud\bigquery\job\query.py:2170: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  query_result = wait_for_query(self, progress_bar_type, max_results=max_results)


""


In [ ]:
%%bigquery
SELECT * FROM `views_fordashboard.avg_fare_analysis` LIMIT 100;

-- Question 9: What is the proportion of different payment types (credit card, cash, etc.), and has it changed over time?

In [5]:
%%bigquery
CREATE OR REPLACE VIEW `views_fordashboard.payment_type_trends` AS
WITH payment_summary AS (
    SELECT 
        DATE(t.tpep_pickup_datetime) AS trip_date,
        EXTRACT(YEAR FROM t.tpep_pickup_datetime) AS year,
        EXTRACT(MONTH FROM t.tpep_pickup_datetime) AS month,
        EXTRACT(WEEK FROM t.tpep_pickup_datetime) AS week,
        t.payment_type,
        COUNT(*) AS total_trips
    FROM `yellow-taxi-trips-2026.transformed_data.cleaned_and_filtered` t
    GROUP BY trip_date, year, month, week, t.payment_type
),
payment_proportion AS (
    SELECT 
        trip_date,
        year,
        month,
        week,
        payment_type,
        total_trips,
        SUM(total_trips) OVER (PARTITION BY trip_date) AS daily_total_trips,
        ROUND(100 * total_trips / SUM(total_trips) OVER (PARTITION BY trip_date), 2) AS payment_percentage
    FROM payment_summary
)
SELECT 
    trip_date,
    year,
    month,
    week,
    CASE 
        WHEN payment_type = 1 THEN 'Credit Card'
        WHEN payment_type = 2 THEN 'Cash'
        WHEN payment_type = 3 THEN 'No Charge'
        WHEN payment_type = 4 THEN 'Dispute'
        WHEN payment_type = 5 THEN 'Unknown'
    END AS payment_method,
    total_trips,
    daily_total_trips,
    payment_percentage
FROM payment_proportion;

c:\Users\EDITH ADJE\Documents\GCP Project\.venv\Lib\site-packages\google\cloud\bigquery\job\query.py:2170: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  query_result = wait_for_query(self, progress_bar_type, max_results=max_results)


""


In [ ]:
SELECT * FROM `views_fordashboard.payment_type_trends` LIMIT 100;

-- Question 10: How often do passengers tip, and what factors (time of day, borough, fare amount) influence tip amounts?

In [6]:
%%bigquery
CREATE OR REPLACE VIEW `views_fordashboard.tipping_behavior_analysis` AS
SELECT 
    DATE(t.tpep_pickup_datetime) AS trip_date,
    EXTRACT(YEAR FROM t.tpep_pickup_datetime) AS year,
    EXTRACT(MONTH FROM t.tpep_pickup_datetime) AS month,
    EXTRACT(HOUR FROM t.tpep_pickup_datetime) AS pickup_hour,
    pz.Borough AS pickup_borough,
    dz.Borough AS dropoff_borough,
    COUNT(*) AS total_trips,
    SUM(CASE WHEN t.tip_amount > 0 THEN 1 ELSE 0 END) AS tipped_trips,
    ROUND(100 * SUM(CASE WHEN t.tip_amount > 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS tip_frequency_percentage,
    ROUND(AVG(t.tip_amount), 2) AS avg_tip_amount,
    ROUND(AVG(t.total_amount), 2) AS avg_total_fare,
    ROUND(AVG(t.fare_amount), 2) AS avg_fare,
    ROUND(AVG(t.tip_amount / NULLIF(t.total_amount, 0)), 4) * 100 AS avg_tip_percentage
FROM `yellow-taxi-trips-2026.transformed_data.cleaned_and_filtered` t
JOIN `yellow-taxi-trips-2026.raw_yellowtrips.taxi_zone` pz 
    ON t.PULocationID = pz.LocationID
JOIN `yellow-taxi-trips-2026.raw_yellowtrips.taxi_zone` dz 
    ON t.DOLocationID = dz.LocationID
WHERE t.payment_type = 1  -- Only credit card payments (cash tips are not recorded)
GROUP BY trip_date, year, month, pickup_hour, pickup_borough, dropoff_borough;

c:\Users\EDITH ADJE\Documents\GCP Project\.venv\Lib\site-packages\google\cloud\bigquery\job\query.py:2170: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  query_result = wait_for_query(self, progress_bar_type, max_results=max_results)


""


In [ ]:
%%bigquery
SELECT * FROM `views_fordashboard.tipping_behavior_analysis` LIMIT 100;

-- Question 11: How much revenue is generated from additional charges (MTA tax, congestion surcharge, airport fees), and has it changed over time?

In [7]:
%%bigquery
CREATE OR REPLACE VIEW `views_fordashboard.additional_charges_revenue` AS
SELECT 
    DATE(t.tpep_pickup_datetime) AS trip_date,
    EXTRACT(YEAR FROM t.tpep_pickup_datetime) AS year,
    EXTRACT(MONTH FROM t.tpep_pickup_datetime) AS month,
    COUNT(*) AS total_trips,
    ROUND(SUM(t.MTA_tax), 2) AS total_MTA_tax,
    ROUND(SUM(t.congestion_surcharge), 2) AS total_congestion_surcharge,
    ROUND(SUM(t.airport_fee), 2) AS total_airport_fees,
    ROUND(SUM(t.MTA_tax + t.congestion_surcharge + t.airport_fee), 2) AS total_additional_revenue,
    ROUND(AVG(t.MTA_tax + t.congestion_surcharge + t.airport_fee), 2) AS avg_additional_charge_per_trip
FROM `yellow-taxi-trips-2026.transformed_data.cleaned_and_filtered` t
GROUP BY trip_date, year, month;

c:\Users\EDITH ADJE\Documents\GCP Project\.venv\Lib\site-packages\google\cloud\bigquery\job\query.py:2170: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  query_result = wait_for_query(self, progress_bar_type, max_results=max_results)


""


In [ ]:
%%bigquery
SELECT * FROM `views_fordashboard.additional_charges_revenue` LIMIT 100;